# Begin

Courtesy: https://docs.cleanrl.dev/rl-algorithms/dqn/#dqn_ataripy

In [36]:
# @launchit.collected

In [37]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter # @launchit.collect
import random
import datetime
import json
import pprint
import re
import uuid
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import Dataset, DataLoader

import gymnasium as gym
import ale_py
import av

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect

from cleanrl.cleanrl_utils.atari_wrappers import (  # isort:skip
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)
from cleanrl.cleanrl_utils.buffers import ReplayBuffer

import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter
from logging_utils import *
from artifact_registry import *
from torch_helpers import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

# Init

In [38]:
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()
    
def create_config():
    config = namedtuple('Config', 
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                        'self_fname, self_name, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, exec_mode, is_interactive')(
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        self_fname=None,
        self_name=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf)['jupyter_session']
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
            config = config._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)
    
    config = config._replace(is_interactive=config.exec_mode != ExecMode.LAUNCH_MODULE)    
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    return config

In [39]:
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.17_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/17_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/17_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/17_rl',
 'self_fname': '/home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_01.ipynb',
 'self_name': '17b_dqn_atari_01',
 'subproject_name': '17_rl',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [40]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    # System params
    random_seed: int = None
    torch_deterministic: bool = None

    # Environment params
    env_id: str = None
    envs_count: int = None # the number of parallel game environments

    # Agent params
    # ...

    # RL params
    gamma: float = None # the discount factor gamma

    # Training procedure params (DQN related) 
    rollouts_count: int = None 
    steps_count: int = None # steps per rollout
    warmup_rollouts_count: int = None # number of rollouts to run before the very first update of an Agent's weights
    oracle_rollouts_period: int = None # number of rollouts to run for Agent's weights are moved to Oracle (a-la stabilization period)
    start_e: float = None # the starting epsilon for exploration
    end_e: float = None # the ending epsilon for exploration
    exploration_fraction: float = None # the fraction of `total-timesteps` it takes from start_e to go end_e
    replay_buffer_size: int = None
    tau: float = None # the Orcale weights' update rate
    capture_video: str = None # video capture policy depending on rollouts, e.g. every(200)

    # Optimization params
    batch_size: int = None
    optimizer: str = None
    learn_rate: str = None

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()
HP.random_seed = 42

# Launch

## LaunchState

In [41]:
@dataclass(slots=True)
class LaunchState:
    envs: list = None
    agent: object = None
    oracle: object = None

## new_artifact_registry

In [42]:
def new_artifact_registry(is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ArtifactRegistry(CONFIG.model_group_uri)

## new_summary_writer

In [43]:
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Create

In [44]:
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', HP.launch_id)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    random.seed(HP.random_seed)
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if HP.torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()
artifact_registry = new_artifact_registry()
artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    
meta = dict(
    optuna_trial_number=getattr(optuna_trial, 'number', None),
    hypers=HP._asdict(), 
    config=CONFIG._asdict(), 
)

with io.StringIO() as b:
    json.dump(meta, b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = lc.name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

LS = LaunchState()

Random seed=42
Tensorboard run=17b_dqn_atari_01/0


# Environment

## create_env

In [45]:
def create_env(env_id, video_dir_name=None, random_seed=None, is_auto_reset=False):
    if video_dir_name is not None:
        env = gym.make(env_id, render_mode='rgb_array')
        env = gym.wrappers.RecordVideo(env, video_dir_name, episode_trigger=lambda episode_id: episode_id == 0)
    else:
        env = gym.make(env_id)
        
    env = gym.wrappers.RecordEpisodeStatistics(env)
    env = NoopResetEnv(env, noop_max=30)
    env = MaxAndSkipEnv(env, skip=4)
    env = EpisodicLifeEnv(env) 
    
    if 'FIRE' in env.unwrapped.get_action_meanings():
        env = FireResetEnv(env)

    if is_auto_reset:
        env = gym.wrappers.Autoreset(env)
        
    env = ClipRewardEnv(env)
    # Following three directly influence envs.single_observation_space
    env = gym.wrappers.ResizeObservation(env, (84, 84))
    env = gym.wrappers.GrayscaleObservation(env)
    env = gym.wrappers.FrameStackObservation(env, 4)

    if random_seed is not None:
        env.action_space.seed(random_seed) # req-d for random sampling from action space when there are multiple envs
        
    return env

## Configure 

In [46]:
# @launchit.disable
# @launchit.collect
HP.env_id = "BreakoutNoFrameskip-v4"
HP.envs_count = 8 # the number of parallel game environments
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': None,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': None,
 'rollouts_count': None,
 'steps_count': None,
 'warmup_rollouts_count': None,
 'oracle_rollouts_period': None,
 'start_e': None,
 'end_e': None,
 'exploration_fraction': None,
 'replay_buffer_size': None,
 'tau': None,
 'capture_video': None,
 'batch_size': None,
 'optimizer': None,
 'learn_rate': None}


## Create

In [47]:
env_factory = lambda env_id, random_seed: lambda: create_env(env_id, random_seed=random_seed)
LS.envs = gym.vector.SyncVectorEnv(
    [env_factory(HP.env_id, HP.random_seed + i) for i in range(HP.envs_count)],
    autoreset_mode=gym.vector.vector_env.AutoresetMode.NEXT_STEP, # https://farama.org/Vector-Autoreset-Mode
)
assert isinstance(LS.envs.single_action_space, gym.spaces.Discrete), "only discrete action space is supported"
LOG(f'{LS.envs.single_observation_space=}, {LS.envs.single_action_space=}, {LS.envs.metadata=}')

LS.envs.single_observation_space=Box(0, 255, (4, 84, 84), uint8), LS.envs.single_action_space=Discrete(4), LS.envs.metadata={'render_modes': ['human', 'rgb_array'], 'render_fps': 30, 'autoreset_mode': <AutoresetMode.NEXT_STEP: 'NextStep'>}


# Agent

## Agent

In [48]:
class Agent(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = 512
        actions_count: int = None 
    
    def __init__(self, params):
        super().__init__()
        self.params = params
        # TODO: use _layer_init
        self.network = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, self.params.d_model),
            nn.ReLU(),
            nn.Linear(self.params.d_model, self.params.actions_count),
        )

    def forward(self, x): 
        return self.network(x / 255.0)

    @staticmethod
    def _layer_init(layer, std=np.sqrt(2), bias_const=0.0):
        torch.nn.init.orthogonal_(layer.weight, std)
        torch.nn.init.constant_(layer.bias, bias_const)
        return layer

## Smoke test

In [49]:
# @launchit.disable
ap = Agent.Params(
    actions_count=10,
)
agent = Agent(ap)
agent = agent.to(CONFIG.cuda_device)
print(agent)
params_count = sum(p.numel() for p in agent.parameters())
print(f'{params_count=:_}')
probe_batch = torch.zeros((2, 4, 84, 84)).to(CONFIG.cuda_device)
print(f'{probe_batch.shape=}')
r = agent(probe_batch)
print(f'{r.shape=}')

Agent(
  (network): Sequential(
    (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=512, bias=True)
    (8): ReLU()
    (9): Linear(in_features=512, out_features=10, bias=True)
  )
)
params_count=1_689_258
probe_batch.shape=torch.Size([2, 4, 84, 84])
r.shape=torch.Size([2, 10])


## Quick play test

In [50]:
# @launchit.disable
ap = Agent.Params(
    actions_count=LS.envs.single_action_space.n.item(),
)
agent = Agent(ap)
agent = agent.to(CONFIG.cuda_device)
env = create_env(HP.env_id, is_auto_reset=True)
obs, _ = env.reset(seed=HP.random_seed)
device = next(iter(agent.parameters())).device

with eval_guard(agent):
    with torch.no_grad():
        for step in tqdm(range(0, 1000)): 
            obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
            logits = agent(obs)[0]
            action = torch.argmax(logits).cpu().item()
            obs, reward, terminated, truncated, info = env.step(action)

            if terminated or truncated:
                if env.get_wrapper_attr('was_real_done'):
                    assert 'episode' in info
                    
                    if 'episode' in info: # 'episode' is a default stats_key for RecordEpisodeStatistics
                        episode_stats = info['episode']
                        print(f'{step:05}', episode_stats)
                else:
                    assert not 'episode' in info
            else:
                assert not 'episode' in info

  0%|          | 0/1000 [00:00<?, ?it/s]

00118 {'r': 0.0, 'l': 524, 't': 0.316666}
00238 {'r': 0.0, 'l': 542, 't': 0.313708}
00358 {'r': 0.0, 'l': 531, 't': 0.3174}
00478 {'r': 0.0, 'l': 535, 't': 0.31559}
00598 {'r': 0.0, 'l': 514, 't': 0.309266}
00718 {'r': 0.0, 'l': 528, 't': 0.317912}
00838 {'r': 0.0, 'l': 524, 't': 0.31821}
00958 {'r': 0.0, 'l': 540, 't': 0.312884}


## Configure

In [51]:
# @launchit.disable
# @launchit.collect_1
# ...
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': None,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': None,
 'rollouts_count': None,
 'steps_count': None,
 'warmup_rollouts_count': None,
 'oracle_rollouts_period': None,
 'start_e': None,
 'end_e': None,
 'exploration_fraction': None,
 'replay_buffer_size': None,
 'tau': None,
 'capture_video': None,
 'batch_size': None,
 'optimizer': None,
 'learn_rate': None}


## Create

In [52]:
# @launchit.disable_2
ap = Agent.Params(
    actions_count=LS.envs.single_action_space.n.item()
)
LS.agent = Agent(ap).to(CONFIG.cuda_device)
LS.oracle = Agent(ap).to(CONFIG.cuda_device)
LS.oracle.load_state_dict(LS.agent.state_dict())

<All keys matched successfully>

# Video

## create_capture_video_policy

In [53]:
def create_capture_video_policy(capture_video):
    def inner_create_capture_video_policy(policy_name, every_rollouts_count):
        if policy_name == 'every':
            def every_thunk(rollout):
                return (rollout % every_rollouts_count) == 0

            return every_thunk
        
        assert False, f'Unsupported capture_video="{module_name}"'
            
    ump = hp_parse_universal_module(capture_video)
    return inner_create_capture_video_policy(ump.module_name, *ump.args, **ump.kwargs)

## get_fresh_video_dir_name

In [54]:
def get_fresh_video_dir_name():
    lc = HP.launch_component()
    timestamp = datetime.datetime.now().strftime("%Y.%m.%d-%H:%M:%S.%f") # generate unique dir name in order to shut up RecordVideo from complaining
    return os.path.join(CONFIG.run_path, f'video-{lc.name}-launch{lc.version}-{timestamp}')

## capture_video_of_test_rollout

In [55]:
def capture_video_of_test_rollout(agent, max_steps_count=10_000, video_dir_name=None):
    video_dir_name = lu.coalesce(video_dir_name, lambda: get_fresh_video_dir_name())
    assert video_dir_name is not None
    env = create_env(HP.env_id, video_dir_name=video_dir_name, is_auto_reset=True)
    assert env.action_space.n.item() == agent.params.actions_count
    obs, _ = env.reset(seed=HP.random_seed)
    device = next(iter(agent.parameters())).device
    
    with eval_guard(agent):
        with torch.no_grad():
            for step in range(0, max_steps_count): 
                obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
                logits = agent(obs)[0]
                action = torch.argmax(logits).cpu().item()
                obs, reward, terminated, truncated, info = env.step(action)
                
                if (terminated or truncated) and env.get_wrapper_attr('was_real_done'):
                    break

    game_meta = dict(
        reward=env.get_wrapper_attr('episode_returns'),
        frames_count=env.get_wrapper_attr('episode_lengths'),
        steps_count=step + 1,
    )
    
    env.close() # this forces video recording to complete and write video file
    
    video_fnames = list(filter(lambda fn: os.path.isfile(os.path.join(video_dir_name, fn)), os.listdir(video_dir_name)))
    assert len(video_fnames) == 1, len(video_fnames)
    video_fname = os.path.join(video_dir_name, video_fnames[0])
    video_meta = {}
    
    with open(video_fname, 'rb') as f:
        container = av.open(f)
        video_meta['fps'] = float(container.streams.video[0].average_rate)
        video_stream = container.streams.video[0]
        video_meta['duration'] = float(video_stream.duration * video_stream.time_base)
        
    return video_fname, dict(game=game_meta, video=video_meta)

In [56]:
# @launchit.disable
capture_video_of_test_rollout(LS.agent, max_steps_count=2000)

('/home/misha/dev/mine/neurolab/run/17_rl/video-17b_dqn_atari_01-launch0-2026.04.30-10:28:17.069524/rl-video-episode-0.mp4',
 {'game': {'reward': 0.0, 'frames_count': 524, 'steps_count': 119},
  'video': {'fps': 30.0, 'duration': 17.5}})

# TRAIN

## Configure

In [59]:
# @launchit.disable
# @launchit.collect

# RL params
HP.gamma = 0.99 # the discount factor gamma

# Training procedure params (DQN related) 
HP.rollouts_count = 60_000
HP.steps_count = 4 # number of steps to run in each environment per policy rollout
HP.warmup_rollouts_count = 1 # number of rollouts to run before the very first update of an Agent's weights
HP.oracle_rollouts_period = 250 # number of rollouts to run for Agent's weights are moved to Oracle (a-la stabilization period)
HP.start_e = 1 # the starting epsilon for exploration
HP.end_e = 0.01 # the ending epsilon for exploration
HP.exploration_fraction = 0.1 # the fraction of `total-timesteps` it takes from start_e to go end_e
HP.replay_buffer_size = 100_000
HP.tau = 1 # the Orcale weights' update rate
HP.capture_video = 'every(10000)' # video capture policy depending on rollouts, e.g. every(200)

# Optimization params
HP.batch_size = 32
HP.optimizer = 'Adam()'
HP.learn_rate = 1e-4

# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': None,
 'env_id': 'BreakoutNoFrameskip-v4',
 'envs_count': 8,
 'gamma': 0.99,
 'rollouts_count': 60000,
 'steps_count': 4,
 'warmup_rollouts_count': 1,
 'oracle_rollouts_period': 250,
 'start_e': 1,
 'end_e': 0.01,
 'exploration_fraction': 0.1,
 'replay_buffer_size': 100000,
 'tau': 1,
 'capture_video': 'every(10000)',
 'batch_size': 32,
 'optimizer': 'Adam()',
 'learn_rate': 0.0001}


## Create

In [58]:
ump = hp_parse_universal_module(HP.optimizer)
assert not ump.args
lr_params = hp_parse_learn_rate(HP.learn_rate)
optimizer = getattr(torch.optim, ump.module_name)(LS.agent.parameters(), lr=lr_params.learn_rate, **ump.kwargs)
capture_video_policy = create_capture_video_policy(HP.capture_video)

## Train

In [62]:
global_step = 0
global_steps_count = HP.rollouts_count * HP.envs_count * HP.steps_count
start_time = time.time()
rb = ReplayBuffer(
    buffer_size=HP.replay_buffer_size * HP.envs_count, 
    observation_space=LS.envs.single_observation_space,
    action_space=LS.envs.single_action_space,
    device=CONFIG.cuda_device,
    optimize_memory_usage=True,
    handle_timeout_termination=False,
    n_envs=HP.envs_count,
)
obs, _ = LS.envs.reset(seed=HP.random_seed)
last_episode_rewards = np.zeros(HP.envs_count)
last_episode_lengths = np.zeros(HP.envs_count)

def linear_schedule(start_e: float, end_e: float, duration: int, t: int):
    slope = (end_e - start_e) / duration
    return max(slope * t + start_e, end_e)

for rollout in tqdm(range(HP.rollouts_count)): # no dry run (i.e. no +1) because we have warmups period which plays the same role as dry run
    for step in range(0, HP.steps_count):
        global_step += HP.envs_count
        epsilon = linear_schedule(HP.start_e, HP.end_e, HP.exploration_fraction * global_steps_count, global_step)
        
        if RNG.random() < epsilon:
            actions = np.array([LS.envs.single_action_space.sample() for _ in range(LS.envs.num_envs)])
        else:
            q_values = LS.agent(torch.Tensor(obs).to(CONFIG.cuda_device))
            actions = torch.argmax(q_values, dim=1).cpu().numpy()

        # TRY NOT TO MODIFY: execute the game and log data.
        # Note: truncation may happen due to max_num_frames_per_episode=108_000 frames limit in ALE
        # (30 minutes of real-time play, assuming the standard 60 frames per second)
        next_obs, rewards, terminations, truncations, infos = LS.envs.step(actions)

        # TRY NOT TO MODIFY: save data to reply buffer; handle `final_observation`
        real_next_obs = next_obs.copy()
        dones = np.logical_or(terminations, truncations)
        rb.add(obs, real_next_obs, actions, rewards, dones, infos)

        # TRY NOT TO MODIFY: CRUCIAL step easy to overlook
        obs = next_obs
        
        if 'episode' in infos: # 'episode' is a default stats_key for RecordEpisodeStatistics
            episode_stats = infos['episode']
            last_episode_rewards[episode_stats['_r']] = episode_stats['r'][episode_stats['_r']]
            last_episode_lengths[episode_stats['_l']] = episode_stats['l'][episode_stats['_l']]

    if rollout >= HP.warmup_rollouts_count:
        data = rb.sample(HP.batch_size)
        assert False
        
        with torch.no_grad():
            target_max, _ = LS.oracle(data.next_observations).max(dim=1)
            td_target = data.rewards.flatten() + HP.gamma * target_max * (1 - data.dones.flatten())

        # data.actions.shape = [batch_size,1], gather will pick from Agent's logits values by indices=data.actions,
        # while squeeze removes first dimension
        old_val = LS.agent(data.observations).gather(1, data.actions).squeeze()

        loss = F.mse_loss(old_val, td_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # update Oracle weights
        if (rollout % HP.oracle_rollouts_period) == 0:
            for oracle_param, agent_param in zip(LS.oracle.parameters(), LS.agent.parameters()):
                oracle_param.data.copy_(HP.tau * agent_param.data + (1.0 - HP.tau) * oracle_param.data)
                
        summary_writer.add_scalar("losses/td_loss", loss.item(), global_step)
        summary_writer.add_scalar("losses/q_values", old_val.mean().item(), global_step)
        
    summary_writer.add_scalar("charts/sps", int(global_step / (time.time() - start_time)), global_step)
    summary_writer.add_scalar("charts/episodic_return", last_episode_rewards.mean(), global_step)
    summary_writer.add_scalar("charts/episodic_length", last_episode_lengths.mean(), global_step)

    if capture_video_policy(rollout):
        video_fname, video_meta = capture_video_of_test_rollout(LS.agent, video_dir_name=get_fresh_video_dir_name(), max_steps_count=5000)
        _, video_fname_ext = os.path.splitext(video_fname)
        ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
        remote_video_fname = f'{ts}-{rollout:06}.{video_fname_ext.lstrip('.')}'
        summary_writer.add_file(video_fname, remote_video_fname)
        summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
        ref_text = f'<a href="http://tensorboard-videos:6007/{summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
        summary_writer.add_text('videos', ref_text, global_step)

    summary_writer.flush()

  0%|          | 0/60000 [00:00<?, ?it/s]

AssertionError: 

## Save

In [ ]:
# @launchit.disable_2
artifact_registry = new_artifact_registry()
lc = HP.launch_component()

with io.BytesIO() as b:
    torch.save(LS.agent.state_dict(), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)

with io.StringIO() as b:
    json.dump(dataclasses.asdict(LS.agent.params), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

# LaunchIt!

## TRAIN

In [18]:
# @launchit.disable
launchit_t0 = time.time()

In [19]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    artifact_registry_obj = new_artifact_registry(is_real=True)
    artifact_registry_obj.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=component_version, expandvars=expandvars, collect_inds=[1], disable_inds=[1])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=8
Creating /home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_01-launch8.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/17_rl/17b_dqn_atari_01-launch8.ipynb"


## Optuna (model selection)

### Templates

In [45]:
# @launchit.disable
# @launchit.collect_3
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP = Hyperparameters()
            HP.random_seed = 42
            assert False
        case _:
            assert False, f'Unsupported {study_serial=}'            

### Unleash

In [ ]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL:
            return ['minimize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 1
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 1
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=4, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")